In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F # (활성화 함수 등을 포함)
import numpy as np
import pandas as pd
import sys

In [3]:
df = pd.read_csv('synthetic_orange_data_2000_kde.csv')
df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'synthetic_orange_data_2000_kde.csv'

In [1]:
try:
    df = pd.read_csv('orange_df.csv')
except FileNotFoundError:
    print("Error: orange_df.csv not found. Please ensure the file is in the same directory as the script.")
    sys.exit() # exit() 대신 sys.exit() 권장

target_variable = '평균 당도'

if target_variable not in df.columns:
    print(f"Error: Target variable '{target_variable}' not found in the DataFrame columns. Available columns: {df.columns.tolist()}")
    sys.exit()

df.dropna(subset=[target_variable], inplace=True)
numeric_df = df.select_dtypes(include=np.number)

if target_variable not in numeric_df.columns:
    print(f"Error: Target variable '{target_variable}' is not numeric and cannot be used for correlation.")
    sys.exit()

# 상관관계 계산
correlations_series = numeric_df.corr()[target_variable]

# 피처에서 제외할 변수 리스트
cols_to_exclude = ['당도1', '당도2', '산도1', '산도2', '평균 산도', '율리우스일', '당도 산도 비율', 'Citric acid (%)', target_variable]
existing_cols_to_exclude = [col for col in cols_to_exclude if col in correlations_series.index]
correlations = correlations_series.drop(existing_cols_to_exclude)

# 0.2 '이상'인 변수들만 사용
selected_features = correlations[np.abs(correlations) >= 0.2].index.tolist()
drop_columns = "평균 무게"  # 제외하고자 하는 컬럼 작성, 없을 시 주석 처리
selected_features = [item for item in selected_features if item not in drop_columns]

if not selected_features:
    print("No numeric features found with absolute correlation to '당도' >= 0.2. Cannot build a model.")
    sys.exit()

print(f"Selected features based on correlation with '{target_variable}' (abs correlation >= 0.2):")
for feature in selected_features:
    print(f"- {feature}: {correlations[feature]:.4f}")

X = df[selected_features]
y = df[target_variable]

# X의 NaN을 처리한 후, 해당 인덱스를 사용해 y를 필터링
X = X.dropna()
y = y[X.index]

if X.empty:
    print("After dropping NaN values, no valid data remains for selected features. Cannot build a model.")
    sys.exit()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

NameError: name 'pd' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F # (활성화 함수 등을 포함)

# 1. DNN 모델 클래스 정의
class RegressionDNN(nn.Module):

    # __init__ (초기화): 모델의 '층(Layer)'들을 정의하는 곳
    def __init__(self, input_dim):
        """
        input_dim (int): 입력 특성(feature)의 개수.
                       (예: 5개의 특성을 사용하면 input_dim=5)
        """
        super(RegressionDNN, self).__init__() # 부모 클래스(nn.Module) 초기화

        # 층(Layer)을 순차적으로 쌓기 위해 nn.Sequential을 사용하면 편리합니다.
        self.network = nn.Sequential(
            # 1번째 은닉층 (Hidden Layer 1)
            # input_dim개의 입력을 받아 64개로 출력
            nn.Linear(input_dim, 64),
            nn.ReLU(),         # 활성화 함수
            nn.Dropout(0.3),   # 30%를 끔 (과적합 방지)

            # 2번째 은닉층 (Hidden Layer 2)
            # 64개를 받아 32개로 출력
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # 3번째 은닉층 (Hidden Layer 3)
            # 32개를 받아 16개로 출력
            nn.Linear(32, 16),
            nn.ReLU(),

            # 출력층 (Output Layer)
            # 16개를 받아 1개로 출력 (이 1개가 '평균 당도' 예측값)
            nn.Linear(16, 1)
        )

    # forward: 데이터가 모델을 통과하는 순서를 정의하는 곳
    def forward(self, x):
        """
        x: 모델에 입력되는 데이터 (배치)
        """
        # nn.Sequential을 사용했으므로, network에 x를 통과시키기만 하면 됨.
        return self.network(x)

# --- (참고) 모델 사용 예시 ---
input_features = 5

# 3. 모델 객체 생성
model = RegressionDNN(input_dim=input_features)

# 4. (테스트) 임의의 데이터(1개 샘플, 5개 특성)로 예측해보기
# 데이터는 Pytorch Tensor 형태여야 함
test_input = torch.randn(1, input_features)
prediction = model(test_input)

print(f"모델 구조:\n {model}")
print(f"\n입력 데이터 (형태: {test_input.shape}):\n {test_input}")
print(f"예측값 (형태: {prediction.shape}):\n {prediction}")